# Border Proximity — Consistency Check & Analysis

Run from the `notebooks/` directory, or open from repo root via Jupyter Lab/Notebook.

**Prerequisites:** Run both pipeline scripts from the repo root:

```bash
python src/4_osm_clip_and_aggregate.py
python src/5_border_proximity.py
```

**Inputs:**

- `../data/outputs/osm_roads_by_class_year.csv` — aggregated road length by municipality / zone / fclass / year
- `../data/outputs/osm_roads_border_proximity.csv` — segment-level road data with binary proximity flags (`in_25m`, `in_50m`, `in_100m`)

**Sections:**

1. Total-length agreement between the two files (sanity check)
2. Share of road length near municipal borders — overview by year
3. Border proximity by zone type
4. Border proximity by fclass (top 10 classes)
5. Year-over-year stability of border proximity shares

## 0. Setup & data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

CLASS_YEAR_PATH = "../data/outputs/osm_roads_by_class_year.csv"
PROX_PATH = "../data/outputs/osm_roads_border_proximity.csv"

ref = pd.read_csv(CLASS_YEAR_PATH)
ref["mpio_id"] = ref["mpio_id"].astype(str).str.zfill(5)
ref["total_length_km"] = ref["total_length_m"] / 1000

prox = pd.read_csv(PROX_PATH)
prox["mpio_id"] = prox["mpio_id"].astype(str).str.zfill(5)
prox["length_km"] = prox["length"] / 1000

years = sorted(prox["year"].unique())
zone_types = ["cabecera", "centro_poblado", "rural"]

ZONE_COLORS = {
    "cabecera": "#4C72B0",
    "centro_poblado": "#55A868",
    "rural": "#C44E52",
}
BUFFER_COLORS = {
    "in_25m": "#e41a1c",
    "in_50m": "#ff7f00",
    "in_100m": "#4daf4a",
    "outside": "#377eb8",
}

print(f"ref  rows : {len(ref):,}  |  years: {sorted(ref['year'].unique())}")
print(f"prox rows : {len(prox):,}  |  years: {years}")
print(f"Municipalities in ref : {ref['mpio_id'].nunique()}")
print(f"Municipalities in prox: {prox['mpio_id'].nunique()}")
prox.head()

## 1. Total-length agreement (sanity check)

Because the 4 proximity rings partition each municipal polygon without overlap, summing all sub-segment lengths in `osm_roads_border_proximity.csv` for a given `(mpio_id, year)` must equal the total in `osm_roads_by_class_year.csv`.

The scatter plot below plots one point per `(mpio_id, year)` pair. All points should fall on the *y = x* diagonal.

In [ ]:
ref_totals = (
    ref.groupby(["mpio_id", "year"])["total_length_km"].sum().reset_index()
    .rename(columns={"total_length_km": "ref_km"})
)
prox_totals = (
    prox.groupby(["mpio_id", "year"])["length_km"].sum().reset_index()
    .rename(columns={"length_km": "prox_km"})
)
merged = ref_totals.merge(prox_totals, on=["mpio_id", "year"], how="outer")
merged["diff_km"] = (merged["prox_km"] - merged["ref_km"]).abs()

print(f"Total (mpio_id, year) pairs — ref: {len(ref_totals):,}  prox: {len(prox_totals):,}")
print(f"Pairs only in ref : {merged['prox_km'].isna().sum()}")
print(f"Pairs only in prox: {merged['ref_km'].isna().sum()}")
print(f"Max absolute difference (km): {merged['diff_km'].max():.4f}")
print(f"Pairs with |diff| > 1 m     : {(merged['diff_km'] > 0.001).sum()}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    merged["ref_km"], merged["prox_km"],
    s=4, alpha=0.3, color="#4C72B0", linewidths=0,
)
lim = max(merged["ref_km"].max(), merged["prox_km"].max()) * 1.02
ax.plot([0, lim], [0, lim], color="tomato", linewidth=1, linestyle="--", label="y = x")
ax.set_xlabel("Total road length — class_year CSV (km)")
ax.set_ylabel("Total road length — border_proximity CSV (km)")
ax.set_title("Length agreement per (mpio_id, year) pair")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 2. Border proximity overview — share of road length by buffer zone

What percentage of total road length falls within 25 m / 50 m / 100 m of a municipal boundary, and what share is in the interior (> 100 m)? Shown for each year.

In [ ]:
# Assign each sub-segment to its most specific ring (mutually exclusive)
prox["ring"] = "outside"
prox.loc[prox["in_100m"] & ~prox["in_50m"], "ring"] = "in_100m"
prox.loc[prox["in_50m"]  & ~prox["in_25m"], "ring"] = "in_50m"
prox.loc[prox["in_25m"],                     "ring"] = "in_25m"

ring_order = ["in_25m", "in_50m", "in_100m", "outside"]
ring_labels = ["0–25 m", "25–50 m", "50–100 m", "> 100 m (interior)"]
ring_colors = [BUFFER_COLORS[r] for r in ring_order]

by_year_ring = (
    prox.groupby(["year", "ring"])["length_km"]
    .sum()
    .unstack("ring")
    .reindex(columns=ring_order)
    .fillna(0)
)
by_year_ring_pct = by_year_ring.div(by_year_ring.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute
by_year_ring.plot(
    kind="bar", stacked=True, ax=axes[0],
    color=ring_colors, edgecolor="white", linewidth=0.3,
)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Road length (km)")
axes[0].set_title("Road length by proximity ring — absolute")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(ring_labels, title="Ring", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
sns.despine(ax=axes[0])

# Percentage
by_year_ring_pct.plot(
    kind="bar", stacked=True, ax=axes[1],
    color=ring_colors, edgecolor="white", linewidth=0.3,
)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Share of total road length (%)")
axes[1].set_title("Road length by proximity ring — percentage")
axes[1].tick_params(axis="x", rotation=0)
axes[1].get_legend().remove()
sns.despine(ax=axes[1])

plt.suptitle("Border proximity — all municipalities, all zone types", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nShare by ring (%) per year:")
print(by_year_ring_pct.round(2).to_string())

## 3. Border proximity by zone type

Does the share of roads near municipal boundaries differ between urban (`cabecera`), secondary urban (`centro_poblado`), and rural areas?  
Smaller zones (cabecera) are expected to have a higher share of roads near their boundaries relative to the total zone area.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for ax, zt in zip(axes, zone_types):
    sub = prox[prox["zone_type"] == zt]
    piv = (
        sub.groupby(["year", "ring"])["length_km"]
        .sum()
        .unstack("ring")
        .reindex(columns=ring_order)
        .fillna(0)
    )
    piv_pct = piv.div(piv.sum(axis=1), axis=0) * 100
    piv_pct.plot(
        kind="bar", stacked=True, ax=ax,
        color=ring_colors, edgecolor="white", linewidth=0.3, legend=False,
    )
    ax.set_title(zt.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Year")
    ax.set_ylabel("Share (%)" if ax == axes[0] else "")
    ax.tick_params(axis="x", rotation=0)
    sns.despine(ax=ax)

fig.suptitle("Border proximity share by zone type × year", fontsize=13)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ring_colors]
fig.legend(
    handles, ring_labels,
    title="Ring", bbox_to_anchor=(1.01, 0.85), loc="upper left", fontsize=9,
)
plt.tight_layout()
plt.show()

# Summary: share within 100m by zone type, latest year
latest = years[-1]
within_100m = (
    prox[prox["year"] == latest]
    .groupby("zone_type")
    .apply(lambda g: g.loc[g["in_100m"], "length_km"].sum() / g["length_km"].sum() * 100)
    .rename(f"within_100m_pct ({latest})")
)
print(within_100m.round(2))

## 4. Border proximity by fclass (top 10 classes)

Which road classes are most concentrated near municipal boundaries?  
A high border-proximity share for a class may indicate roads built along administrative limits (e.g. boundary-following paths or tracks) or simply that short segments near borders are more likely to belong to lower-hierarchy classes.

In [ ]:
# Use the reference CSV to identify the top 10 fclasses by total length
top_classes = (
    ref.groupby("fclass")["total_length_km"].sum()
    .nlargest(10).index.tolist()
)

prox_top = prox[prox["fclass"].isin(top_classes)]

fclass_ring = (
    prox_top.groupby(["fclass", "ring"])["length_km"]
    .sum()
    .unstack("ring")
    .reindex(columns=ring_order)
    .fillna(0)
)
fclass_ring_pct = fclass_ring.div(fclass_ring.sum(axis=1), axis=0) * 100

# Sort by share within 100m (in_25m + in_50m + in_100m) descending
fclass_ring_pct["within_100m"] = (
    fclass_ring_pct["in_25m"] + fclass_ring_pct["in_50m"] + fclass_ring_pct["in_100m"]
)
fclass_ring_pct = fclass_ring_pct.sort_values("within_100m", ascending=False)
fclass_ring_pct = fclass_ring_pct.drop(columns="within_100m")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

fclass_ring_pct.plot(
    kind="barh", stacked=True, ax=axes[0],
    color=ring_colors, edgecolor="white", linewidth=0.3,
)
axes[0].set_xlabel("Share of class total (%)")
axes[0].set_ylabel("fclass")
axes[0].set_title("Border proximity share by fclass (all years combined)")
axes[0].legend(ring_labels, title="Ring", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
axes[0].invert_yaxis()
sns.despine(ax=axes[0])

# Absolute lengths stacked bar for comparison with ref totals
ref_by_fclass = (
    ref[ref["fclass"].isin(top_classes)]
    .groupby("fclass")["total_length_km"].sum()
    .reindex(fclass_ring_pct.index)
)
fclass_ring_abs = fclass_ring.reindex(fclass_ring_pct.index)

x = np.arange(len(fclass_ring_pct))
bottom = np.zeros(len(x))
for ring, color, label in zip(ring_order, ring_colors, ring_labels):
    axes[1].bar(x, fclass_ring_abs[ring], bottom=bottom, color=color,
                edgecolor="white", linewidth=0.3, label=label)
    bottom += fclass_ring_abs[ring].values
axes[1].scatter(x, ref_by_fclass.values, color="black", zorder=5, s=40,
                label="Total (class_year CSV)", marker="D")
axes[1].set_xticks(x)
axes[1].set_xticklabels(fclass_ring_pct.index, rotation=30, ha="right")
axes[1].set_ylabel("Road length (km)")
axes[1].set_title("Absolute length by ring — diamonds = reference total")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
sns.despine(ax=axes[1])

plt.suptitle("Border proximity by fclass — top 10 classes", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\nShare within each ring (%) by fclass:")
print(fclass_ring_pct.round(2).to_string())

## 5. Year-over-year stability of border proximity shares

If the proximity shares change substantially year-over-year it may indicate that new OSM edits are not uniformly distributed across the municipality (e.g. a mapping campaign that focused on border areas).  
This section shows the share within 100 m for each year and fclass, alongside the reference total length — allowing both trends to be read together.

In [ ]:
# Share within 100m by fclass × year
within_100m_by_year = (
    prox[prox["fclass"].isin(top_classes)]
    .groupby(["year", "fclass"])
    .apply(lambda g: g.loc[g["in_100m"], "length_km"].sum() / g["length_km"].sum() * 100)
    .unstack("fclass")
    .reindex(columns=top_classes)
    .fillna(0)
)

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

# Top: share within 100m per fclass over time
palette = sns.color_palette("tab10", len(top_classes))
for col, color in zip(within_100m_by_year.columns, palette):
    axes[0].plot(
        within_100m_by_year.index, within_100m_by_year[col],
        marker="o", label=col, color=color, linewidth=2,
    )
axes[0].set_ylabel("% of class length within 100 m of border")
axes[0].set_title("Share of road length within 100 m of municipal boundary by fclass")
axes[0].legend(title="fclass", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
sns.despine(ax=axes[0])

# Bottom: total road length per year from reference CSV (for context)
ref_by_fclass_year = (
    ref[ref["fclass"].isin(top_classes)]
    .groupby(["year", "fclass"])["total_length_km"]
    .sum()
    .unstack("fclass")
    .reindex(columns=top_classes)
    .fillna(0)
)
for col, color in zip(ref_by_fclass_year.columns, palette):
    axes[1].plot(
        ref_by_fclass_year.index, ref_by_fclass_year[col],
        marker="o", label=col, color=color, linewidth=2,
    )
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Total road length (km)")
axes[1].set_title("Reference total road length by fclass (class_year CSV)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
axes[1].legend(title="fclass", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
sns.despine(ax=axes[1])

plt.suptitle("Border proximity stability vs. network growth — top 10 fclasses", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nShare within 100 m (%) by fclass × year:")
print(within_100m_by_year.round(2).to_string())